# TRABALHO FINAL INTEGRADOR - SEMANA 3

## Agrupamento de Dados II: comparação, validação e interpretação

**Projeto:** Detecção de Fraude em Transações Financeiras  
**Base:** Credit Card Fraud Detection  
**Objetivo:** comparar o baseline de agrupamento da Semana 2 com um segundo algoritmo, validar os resultados, interpretar perfis e gerar uma estrutura de saída para uso posterior em Inteligência Computacional.

### Entrega obrigatória desta semana

- Aplicação de um segundo algoritmo de agrupamento.
- Comparação entre os algoritmos.
- Uso de pelo menos duas métricas de avaliação.
- Análise dos parâmetros utilizados.
- Interpretação dos clusters.
- Nomeação dos perfis encontrados.
- Definição de como os clusters serão usados na Inteligência Computacional.
- Geração da variável ou estrutura de saída do agrupamento.

## 1️⃣ CONTEXTO E ESTRATÉGIA

Na Semana 2 foi criado um primeiro baseline com K-Means. Nesta semana, o objetivo é comparar esse baseline com um segundo algoritmo. A proposta preliminar usa **DBSCAN**, pois ele consegue identificar regiões densas e marcar pontos ruidosos como `-1`, o que é útil em um cenário de fraude, onde casos suspeitos podem aparecer como comportamento atípico.

A coluna `Class` não será usada para treinar os agrupamentos. Ela será usada apenas como referência externa para analisar se algum cluster concentra mais transações fraudulentas.

## 2️⃣ IMPORTAÇÕES E CONFIGURAÇÕES

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler, RobustScaler

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
SAMPLE_SIZE = 15000
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14

print('Configurações carregadas com sucesso.')

## 3️⃣ CARREGAMENTO E PREPARAÇÃO DOS DADOS

O carregamento abaixo funciona tanto ao executar o notebook a partir da raiz do projeto quanto de dentro da pasta `notebooks/`.

In [ ]:
data_path = os.path.join('..', 'dados', 'creditcard.csv')

if not os.path.exists(data_path):
    raise FileNotFoundError(f'Arquivo não encontrado: {data_path}')

df = pd.read_csv(data_path)

print(f'Dados carregados de: {data_path}')
print(f'Dimensões: {df.shape[0]:,} linhas x {df.shape[1]} colunas')
display(df.head())

## 4️⃣ SELEÇÃO DE ATRIBUTOS

Para manter o padrão do baseline da Semana 2, serão usados os componentes `V1` a `V28`, `Time` e `Amount`. A única variável excluída da matriz de agrupamento é `Class`, pois ela representa o rótulo real de fraude e deve ser usada apenas para análise externa.

`Class` será separada apenas para validação externa, sem participar do treinamento dos algoritmos.

In [ ]:
feature_cols = [col for col in df.columns if col.startswith('V')] + ['Time', 'Amount']

X = df[feature_cols].copy()
y = df['Class'].copy() if 'Class' in df.columns else None

print('Atributos selecionados:')
print(feature_cols)
print(f'Total de atributos: {len(feature_cols)}')

if y is not None:
    print('\nDistribuição da classe real, usada apenas para análise externa:')
    display(y.value_counts().rename(index={0: 'Legítima', 1: 'Fraude'}).to_frame('quantidade'))

## 5️⃣ NORMALIZAÇÃO / PADRONIZAÇÃO

A escala é obrigatória porque `Amount` tem ordem de grandeza diferente dos componentes PCA. Para esta comparação preliminar:

- `StandardScaler` será usado no K-Means para manter consistência com o baseline.
- `RobustScaler` será usado no DBSCAN para reduzir o impacto de valores extremos em `Amount`.

Essa diferença também será considerada na análise de parâmetros.

In [ ]:
standard_scaler = StandardScaler()
robust_scaler = RobustScaler()

X_standard = standard_scaler.fit_transform(X)
X_robust = robust_scaler.fit_transform(X)

print('Dados padronizados para K-Means:', X_standard.shape)
print('Dados robustamente escalados para DBSCAN:', X_robust.shape)

## 6️⃣ AMOSTRA PARA COMPARAÇÃO

DBSCAN pode ser custoso na base completa. Para a comparação preliminar, será usada uma amostra estratificada quando `Class` estiver disponível. Isso preserva casos de fraude na análise, mesmo com forte desbalanceamento.

In [ ]:
if len(df) > SAMPLE_SIZE:
    if y is not None and y.nunique() == 2:
        sample_index = (
            df.groupby('Class', group_keys=False)
              .apply(lambda part: part.sample(
                  n=min(len(part), max(1, int(SAMPLE_SIZE * len(part) / len(df)))),
                  random_state=RANDOM_STATE
              ))
              .index
        )
        if len(sample_index) < SAMPLE_SIZE:
            remaining = df.index.difference(sample_index)
            extra = np.random.default_rng(RANDOM_STATE).choice(
                remaining,
                size=min(SAMPLE_SIZE - len(sample_index), len(remaining)),
                replace=False
            )
            sample_index = sample_index.union(extra)
    else:
        sample_index = df.sample(SAMPLE_SIZE, random_state=RANDOM_STATE).index
else:
    sample_index = df.index

sample_index = pd.Index(sample_index).sort_values()
df_sample = df.loc[sample_index].copy()
X_standard_sample = X_standard[sample_index]
X_robust_sample = X_robust[sample_index]

print(f'Amostra para comparação: {len(sample_index):,} registros')
if y is not None:
    display(df_sample['Class'].value_counts().rename(index={0: 'Legítima', 1: 'Fraude'}).to_frame('quantidade'))

## 7️⃣ ALGORITMO 1: K-MEANS BASELINE

O K-Means será reconstruído aqui para comparação direta. Ele usa distância euclidiana e exige a definição prévia de `k`. Como ponto preliminar, serão testados alguns valores e escolhido o melhor pela métrica de Silhouette na amostra.

In [ ]:
k_results = []

for k in range(2, 8):
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = model.fit_predict(X_standard_sample)
    k_results.append({
        'k': k,
        'silhouette': silhouette_score(X_standard_sample, labels),
        'davies_bouldin': davies_bouldin_score(X_standard_sample, labels),
        'calinski_harabasz': calinski_harabasz_score(X_standard_sample, labels),
        'inertia': model.inertia_,
    })

k_results_df = pd.DataFrame(k_results).sort_values('silhouette', ascending=False)
display(k_results_df)

best_k = int(k_results_df.iloc[0]['k'])
print(f'Melhor k preliminar por Silhouette: {best_k}')

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
df_sample['cluster_kmeans_semana3'] = kmeans.fit_predict(X_standard_sample)

print('Distribuição dos clusters K-Means:')
display(df_sample['cluster_kmeans_semana3'].value_counts().sort_index().to_frame('quantidade'))

## 8️⃣ ALGORITMO 2: DBSCAN

O DBSCAN depende principalmente de dois parâmetros:

- `eps`: raio máximo para considerar pontos como vizinhos.
- `min_samples`: quantidade mínima de pontos para formar uma região densa.

Pontos classificados como `-1` são considerados ruído. No contexto de fraude, esse grupo merece atenção especial.

In [ ]:
min_samples = max(10, 2 * len(feature_cols))

neighbors = NearestNeighbors(n_neighbors=min_samples)
neighbors_fit = neighbors.fit(X_robust_sample)
distances, _ = neighbors_fit.kneighbors(X_robust_sample)
k_distances = np.sort(distances[:, -1])

candidate_eps = float(np.percentile(k_distances, 95))

plt.figure(figsize=(12, 5))
plt.plot(k_distances)
plt.axhline(candidate_eps, color='red', linestyle='--', label=f'eps sugerido p95 = {candidate_eps:.3f}')
plt.title('Curva k-distance para escolha preliminar do eps')
plt.xlabel('Pontos ordenados por distância')
plt.ylabel(f'Distância ao {min_samples}º vizinho')
plt.legend()
plt.show()

print(f'min_samples escolhido: {min_samples}')
print(f'eps preliminar sugerido: {candidate_eps:.4f}')

In [ ]:
eps_values = [candidate_eps * factor for factor in [0.75, 1.0, 1.25]]
dbscan_results = []

for eps in eps_values:
    dbscan_temp = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1)
    labels = dbscan_temp.fit_predict(X_robust_sample)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_rate = np.mean(labels == -1)
    valid_mask = labels != -1
    
    if n_clusters >= 2 and valid_mask.sum() > n_clusters:
        sil = silhouette_score(X_robust_sample[valid_mask], labels[valid_mask])
        db = davies_bouldin_score(X_robust_sample[valid_mask], labels[valid_mask])
        ch = calinski_harabasz_score(X_robust_sample[valid_mask], labels[valid_mask])
    else:
        sil = np.nan
        db = np.nan
        ch = np.nan
    
    dbscan_results.append({
        'eps': eps,
        'min_samples': min_samples,
        'clusters_sem_ruido': n_clusters,
        'taxa_ruido': noise_rate,
        'silhouette_sem_ruido': sil,
        'davies_bouldin_sem_ruido': db,
        'calinski_harabasz_sem_ruido': ch,
    })

dbscan_results_df = pd.DataFrame(dbscan_results)
display(dbscan_results_df)

In [ ]:
valid_dbscan = dbscan_results_df.dropna(subset=['silhouette_sem_ruido'])

if valid_dbscan.empty:
    best_eps = float(candidate_eps)
else:
    best_eps = float(valid_dbscan.sort_values('silhouette_sem_ruido', ascending=False).iloc[0]['eps'])

dbscan = DBSCAN(eps=best_eps, min_samples=min_samples, n_jobs=-1)
df_sample['cluster_dbscan_semana3'] = dbscan.fit_predict(X_robust_sample)

print(f'eps final escolhido: {best_eps:.4f}')
print('Distribuição dos clusters DBSCAN:')
display(df_sample['cluster_dbscan_semana3'].value_counts().sort_index().to_frame('quantidade'))

## 9️⃣ MÉTRICAS DE AVALIAÇÃO E COMPARAÇÃO

Serão usadas pelo menos duas métricas internas:

- **Silhouette Score:** quanto maior, melhor separação e coesão.
- **Davies-Bouldin:** quanto menor, melhor.
- **Calinski-Harabasz:** quanto maior, melhor.

No DBSCAN, as métricas são calculadas sem os pontos de ruído (`-1`), pois ruído não representa um cluster convencional.

In [ ]:
def clustering_metrics(X_values, labels, ignore_noise=False):
    labels = np.asarray(labels)
    mask = np.ones(len(labels), dtype=bool)
    
    if ignore_noise:
        mask = labels != -1
    
    labels_eval = labels[mask]
    X_eval = X_values[mask]
    n_clusters = len(set(labels_eval))
    
    if n_clusters < 2 or len(labels_eval) <= n_clusters:
        return {
            'clusters_avaliados': n_clusters,
            'silhouette': np.nan,
            'davies_bouldin': np.nan,
            'calinski_harabasz': np.nan,
        }
    
    return {
        'clusters_avaliados': n_clusters,
        'silhouette': silhouette_score(X_eval, labels_eval),
        'davies_bouldin': davies_bouldin_score(X_eval, labels_eval),
        'calinski_harabasz': calinski_harabasz_score(X_eval, labels_eval),
    }

comparison = pd.DataFrame([
    {
        'algoritmo': 'K-Means',
        'parametros': f'k={best_k}, scaler=StandardScaler',
        **clustering_metrics(X_standard_sample, df_sample['cluster_kmeans_semana3']),
        'taxa_ruido': 0.0,
    },
    {
        'algoritmo': 'DBSCAN',
        'parametros': f'eps={best_eps:.4f}, min_samples={min_samples}, scaler=RobustScaler',
        **clustering_metrics(X_robust_sample, df_sample['cluster_dbscan_semana3'], ignore_noise=True),
        'taxa_ruido': float(np.mean(df_sample['cluster_dbscan_semana3'] == -1)),
    },
])

display(comparison)

## 🔟 VISUALIZAÇÃO DOS AGRUPAMENTOS

A visualização em 2D usa PCA apenas para projeção gráfica. Ela não substitui as métricas, mas ajuda a identificar separação, sobreposição e presença de ruído.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_standard_sample)

df_sample['pca_1'] = X_pca[:, 0]
df_sample['pca_2'] = X_pca[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(
    data=df_sample,
    x='pca_1',
    y='pca_2',
    hue='cluster_kmeans_semana3',
    palette='tab10',
    s=12,
    alpha=0.7,
    ax=axes[0]
)
axes[0].set_title('K-Means - projeção PCA')

sns.scatterplot(
    data=df_sample,
    x='pca_1',
    y='pca_2',
    hue='cluster_dbscan_semana3',
    palette='tab10',
    s=12,
    alpha=0.7,
    ax=axes[1]
)
axes[1].set_title('DBSCAN - projeção PCA')

plt.tight_layout()
plt.show()

print(f'Variância explicada pelos 2 componentes: {pca.explained_variance_ratio_.sum():.2%}')

## 1️⃣1️⃣ INTERPRETAÇÃO DOS CLUSTERS

A interpretação abaixo resume cada grupo por quantidade de transações, valor médio/mediano e proporção de fraude. Como `V1` a `V28` são componentes PCA anonimizados, a interpretação prática deve se apoiar principalmente em `Amount`, presença de ruído e concentração relativa de fraude.

In [ ]:
def cluster_profile_table(data, cluster_col):
    agg_dict = {
        'Amount': ['count', 'mean', 'median', 'min', 'max'],
    }
    
    if 'Class' in data.columns:
        agg_dict['Class'] = ['sum', 'mean']
    
    profile = data.groupby(cluster_col).agg(agg_dict)
    profile.columns = ['_'.join(col).strip() for col in profile.columns.values]
    profile = profile.rename(columns={
        'Amount_count': 'quantidade',
        'Amount_mean': 'amount_medio',
        'Amount_median': 'amount_mediano',
        'Amount_min': 'amount_minimo',
        'Amount_max': 'amount_maximo',
        'Class_sum': 'fraudes',
        'Class_mean': 'taxa_fraude',
    })
    
    if 'taxa_fraude' in profile.columns:
        profile['taxa_fraude'] = profile['taxa_fraude'] * 100
    
    return profile.sort_values('quantidade', ascending=False)

perfil_kmeans = cluster_profile_table(df_sample, 'cluster_kmeans_semana3')
perfil_dbscan = cluster_profile_table(df_sample, 'cluster_dbscan_semana3')

print('Perfil dos clusters - K-Means')
display(perfil_kmeans)

print('Perfil dos clusters - DBSCAN')
display(perfil_dbscan)

## 1️⃣2️⃣ NOMEAÇÃO DOS PERFIS

Com base nas tabelas de perfil dos clusters, a interpretação preliminar é:

### K-Means

- **Cluster 2:** perfil crítico de fraude concentrada. Apesar de possuir apenas 13 transações na amostra, concentrou 12 fraudes, com taxa de fraude de 92,31%.
- **Cluster 0:** perfil de risco acima da média, com taxa de fraude de 0,28%.
- **Cluster 3:** perfil de transações de alto valor financeiro, com `Amount` médio de aproximadamente US$ 1.355,74, mas sem fraudes observadas na amostra.
- **Clusters 1 e 4:** perfis transacionais predominantes, com baixo volume relativo de fraude.
- **Cluster 5:** perfil transacional sem fraude observada na amostra.

### DBSCAN

- **Cluster 0:** comportamento transacional dominante, com taxa de fraude muito baixa, aproximadamente 0,014%.
- **Cluster -1:** ruído/anomalia, com 386 transações e taxa de fraude de aproximadamente 5,96%.

Assim, o K-Means é mais útil para nomear perfis múltiplos, enquanto o DBSCAN é mais útil para identificar comportamento atípico com maior concentração de fraude.

In [ ]:
def nomear_perfis(data, cluster_col):
    profiles = cluster_profile_table(data, cluster_col)
    amount_median_global = data['Amount'].median()
    fraud_rate_global = data['Class'].mean() * 100 if 'Class' in data.columns else 0
    names = {}
    
    for cluster_id, row in profiles.iterrows():
        if cluster_id == -1:
            names[cluster_id] = 'Ruido / comportamento atipico'
        elif 'taxa_fraude' in profiles.columns and row['taxa_fraude'] > fraud_rate_global * 2:
            names[cluster_id] = 'Perfil com maior incidencia de fraude'
        elif row['amount_mediano'] > amount_median_global:
            names[cluster_id] = 'Perfil de transacoes de maior valor'
        else:
            names[cluster_id] = 'Perfil transacional predominante'
    
    return names

nomes_kmeans = nomear_perfis(df_sample, 'cluster_kmeans_semana3')
nomes_dbscan = nomear_perfis(df_sample, 'cluster_dbscan_semana3')

df_sample['perfil_kmeans'] = df_sample['cluster_kmeans_semana3'].map(nomes_kmeans)
df_sample['perfil_dbscan'] = df_sample['cluster_dbscan_semana3'].map(nomes_dbscan)

display(pd.DataFrame({
    'cluster_kmeans': list(nomes_kmeans.keys()),
    'perfil_kmeans': list(nomes_kmeans.values()),
}))

display(pd.DataFrame({
    'cluster_dbscan': list(nomes_dbscan.keys()),
    'perfil_dbscan': list(nomes_dbscan.values()),
}))

## 1️⃣3️⃣ DEFINIÇÃO DE USO NA INTELIGÊNCIA COMPUTACIONAL

Uso proposto para as Semanas 4 e 5:

- Usar o rótulo do cluster como nova variável categórica.
- Usar o indicador de ruído do DBSCAN como possível sinal de anomalia.
- Comparar modelos supervisionados com e sem variáveis de agrupamento.
- Avaliar se clusters aumentam recall/precision para fraude sem depender de acurácia.

In [ ]:
saida_clusters = df_sample[[
    'cluster_kmeans_semana3',
    'perfil_kmeans',
    'cluster_dbscan_semana3',
    'perfil_dbscan',
]].copy()

saida_clusters['dbscan_ruido'] = (saida_clusters['cluster_dbscan_semana3'] == -1).astype(int)
saida_clusters['usar_em_ic'] = 'cluster como feature + ruido DBSCAN como sinal de anomalia'

if 'Class' in df_sample.columns:
    saida_clusters['Class'] = df_sample['Class']

display(saida_clusters.head())
print(f'Estrutura de saída criada com {saida_clusters.shape[0]:,} linhas e {saida_clusters.shape[1]} colunas.')

## 1️⃣4️⃣ EXPORTAÇÃO PRELIMINAR

A célula abaixo salva a estrutura de saída em `dados/saida_clusters_semana3.csv`. Como a pasta `dados/` normalmente é ignorada pelo Git, esse arquivo é um artefato local para integração e testes posteriores.

In [ ]:
output_paths = [
    os.path.join('dados', 'saida_clusters_semana3.csv'),
    os.path.join('..', 'dados', 'saida_clusters_semana3.csv'),
]

output_path = next((path for path in output_paths if os.path.exists(os.path.dirname(path))), output_paths[0])
saida_clusters.to_csv(output_path, index=True, index_label='original_index')

print(f'Arquivo de saída salvo em: {output_path}')

## 1️⃣5️⃣ CONCLUSÃO PRELIMINAR

- **Algoritmo com melhor métrica interna:** K-Means. Ele formou 6 clusters avaliáveis e obteve Silhouette de 0,0838, Davies-Bouldin de 2,4227 e Calinski-Harabasz de 617,44. Apesar de o Silhouette ser baixo, o K-Means foi o único algoritmo que gerou múltiplos clusters comparáveis pelas métricas internas.
- **Algoritmo com interpretação mais útil:** os dois algoritmos entregam informações complementares. O K-Means é mais útil para segmentar perfis comportamentais, enquanto o DBSCAN é mais útil para identificar anomalias, pois o grupo de ruído (`-1`) concentrou taxa de fraude muito superior à média.
- **Clusters/perfis mais relevantes:** no K-Means, o cluster 2 é o principal perfil crítico, com 12 fraudes em 13 transações e taxa de fraude de 92,31%. O cluster 0 também merece atenção por ter taxa de fraude acima da média. No DBSCAN, o grupo `-1` representa ruído/anomalia e concentrou 23 fraudes em 386 transações, com taxa de fraude de 5,96%.
- **Problemas encontrados:** o K-Means apresentou Silhouette baixo, indicando sobreposição entre clusters. O DBSCAN, com os parâmetros testados, gerou apenas um cluster principal e um grupo de ruído, impossibilitando o cálculo de Silhouette, Davies-Bouldin e Calinski-Harabasz para múltiplos clusters. Isso mostra que o DBSCAN funcionou melhor como detector de anomalia do que como segmentador de perfis.
- **Uso definido para IC:** usar `cluster_kmeans_semana3` como feature categórica de perfil e criar `dbscan_ruido` como indicador binário de anomalia. Na Semana 4, comparar modelos supervisionados com e sem essas variáveis para verificar se os clusters aumentam o desempenho em AUC-ROC, F1-Score, Recall e Precision.

Conclusão geral: o K-Means deve ser mantido como estrutura principal de segmentação, e o DBSCAN deve ser aproveitado como sinal complementar de transação atípica. Essa combinação é promissora para enriquecer a base de Inteligência Computacional sem usar `Class` no processo de agrupamento.